# Train SLM Router — Qwen3-1.7B Multi-task Fine-tuning (Colab A100)

**Task:** Fine-tune Qwen3-1.7B for dual-output routing + contextual query rewriting
**Hardware:** Google Colab Pro — A100 40GB
**Precision:** bf16 (A100 native) — no quantization during training

**Improvements vs Kaggle T4 version:**
| | Kaggle T4 | Colab A100 |
|---|---|---|
| VRAM | 15.6 GB | 40 GB |
| Precision | fp16 QLoRA (4-bit) | bf16 full weights |
| LoRA rank | r=16 | r=32 |
| Batch size | 4 | 8 |
| Optimizer | adamw_8bit | adamw_torch_fused |

In [ ]:
# Cell 1: Install packages
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "unsloth[colab-new]", "trl>=0.17,<0.19"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Install failed")
print("Packages installed")


In [ ]:
# Cell 2: Mount Google Drive + verify A100
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > A100"
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"bf16: {torch.cuda.is_bf16_supported()}")   # Must be True on A100


In [ ]:
# Cell 3: Imports
# IMPORTANT: unsloth MUST be imported FIRST to apply model patches
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

import json, os
from pathlib import Path
from collections import Counter
import torch
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

import trl
print(f"TRL version: {trl.__version__}")
print(f"Torch:       {torch.__version__}")


In [ ]:
# Cell 4: Configuration

# ── Model ──
BASE_MODEL = "unsloth/Qwen3-1.7B"   # full bf16 — no 4-bit needed on A100

# ── Paths: saved to Drive for persistence across Colab sessions ──
DRIVE_DIR  = Path("/content/drive/MyDrive/hanoi-router-v2")
OUTPUT_DIR = str(DRIVE_DIR / "outputs")
GGUF_DIR   = str(DRIVE_DIR / "gguf")

# ── Data: put multitask_train.jsonl + multitask_val.jsonl in this Drive folder ──
DATA_DIR   = Path("/content/drive/MyDrive/chatbot-hanoi-weather")
TRAIN_FILE = DATA_DIR / "multitask_train.jsonl"
VAL_FILE   = DATA_DIR / "multitask_val.jsonl"

# ── LoRA: r=32 vs r=16 on Kaggle — 2x capacity for rewrite entity grounding ──
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.0

# ── Training ──
MAX_SEQ_LENGTH = 1024
EPOCHS         = 5
BATCH_SIZE     = 8      # A100 40GB — 2x Kaggle T4
GRAD_ACCUM     = 4      # Effective batch = 32 (same as Kaggle version)
LR             = 2e-4
WARMUP_RATIO   = 0.05

# ── Domain constants (must match app/agent/router/config.py) ──
VALID_INTENTS = [
    "current_weather", "hourly_forecast", "daily_forecast", "weather_overview",
    "rain_query", "temperature_query", "wind_query", "humidity_fog_query",
    "historical_weather", "location_comparison", "activity_weather",
    "expert_weather_param", "weather_alert", "seasonal_context", "smalltalk_weather",
]
VALID_SCOPES = ["city", "district", "ward"]


In [ ]:
SYSTEM_PROMPT = """Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON.

## Intents:
- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)
- hourly_forecast: diễn biến CHI TIẾT THEO GIỜ trong ngày (không chỉ hỏi mưa)
- daily_forecast: dự báo NHIỀU NGÀY tới (3 ngày, tuần tới, cuối tuần)
- weather_overview: TỔNG QUAN, tóm tắt thời tiết hôm nay/ngày mai (không hỏi thông số cụ thể)
- rain_query: hỏi CÓ MƯA KHÔNG, mưa bao lâu, xác suất mưa, mưa lúc nào tạnh
- temperature_query: hỏi CỤ THỂ VỀ NHIỆT ĐỘ (bao nhiêu độ, nóng/lạnh thế nào)
- wind_query: hỏi CỤ THỂ VỀ GIÓ (gió mạnh không, hướng gió, tốc độ gió)
- humidity_fog_query: hỏi về ĐỘ ẨM, SƯƠNG MÙ, sương muối
- historical_weather: thời tiết NGÀY/TUẦN TRƯỚC, dữ liệu QUÁ KHỨ
- location_comparison: SO SÁNH thời tiết giữa các quận/phường/địa điểm
- activity_weather: thời tiết có PHÙ HỢP ĐỂ LÀM hoạt động X không (chạy bộ, picnic, du lịch)
- expert_weather_param: thông số KỸ THUẬT chuyên sâu (áp suất, tia UV, điểm sương, tầm nhìn)
- weather_alert: CẢNH BÁO, nguy hiểm, bão, ngập, thay đổi đột ngột
- seasonal_context: SO SÁNH với hôm qua/tuần trước, xu hướng, bất thường theo MÙA
- smalltalk_weather: chào hỏi, ngoài phạm vi (không phải Hà Nội), câu hỏi không liên quan thời tiết

## Scopes:
- city: toàn Hà Nội, hoặc KHÔNG NÓI RÕ địa điểm
- district: nhắc tên QUẬN/HUYỆN (Ba Đình, Cầu Giấy...) hoặc ĐỊA ĐIỂM NỔI TIẾNG thuộc quận (Hồ Gươm→Hoàn Kiếm, Lăng Bác→Ba Đình, Nội Bài→Sóc Sơn...)
- ward: nhắc tên PHƯỜNG/XÃ (Phường Dịch Vọng Hậu, Xã Tiên Dương...)

## Multi-task output (khi có context từ lượt trước):
Nếu được cung cấp context (location/intent từ lượt trước) VÀ câu hỏi hiện tại thiếu địa điểm hoặc dùng đại từ (ở đó, thế còn, còn ..., vậy ...), hãy điền thêm trường "rewritten_query" với câu hỏi đầy đủ ngữ cảnh.

## Output format:
Khi không có context hoặc câu hỏi đầy đủ:
{"intent": "...", "scope": "...", "confidence": 0.92}

Khi có context và cần rewrite:
{"intent": "...", "scope": "...", "confidence": 0.91, "rewritten_query": "Dự báo thời tiết ngày mai ở quận Cầu Giấy như thế nào?"}"""

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(GGUF_DIR).mkdir(parents=True, exist_ok=True)

print(f"Base model:      {BASE_MODEL}")
print(f"LoRA:            r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
# Cell 5: Load base model — Qwen3-1.7B full bf16
# A100 40GB: weights ~3.4GB — plenty of headroom for batch_size=8
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,   # A100 native bf16 — no precision loss
    load_in_4bit=False,     # full weights — better gradient quality than QLoRA
)
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Vocab:       {tokenizer.vocab_size}")
vram  = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM used:   {vram:.1f} GB / {total:.1f} GB after model load")


In [ ]:
# Cell 6: Configure LoRA (r=32)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({trainable/total_p*100:.2f}%)")


In [ ]:
# Cell 7: Chat template setup
# get_chat_template() is SKIPPED — newer Unsloth has padding_side AttributeError on Qwen3.
# Qwen3 tokenizer already has ChatML template built-in; just configure pad token.

# Ensure pad token is set (Qwen3 uses a special PAD token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"   # required for causal LM training

# Smoke test — confirm ChatML template works
test_msgs = [{"role": "system", "content": "Test"}, {"role": "user", "content": "Hello"}]
try:
    preview = tokenizer.apply_chat_template(
        test_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError:
    preview = tokenizer.apply_chat_template(
        test_msgs, tokenize=False, add_generation_prompt=True)
print("Chat template preview:")
print(preview)
print(f"pad_token: {tokenizer.pad_token!r}")
print(f"padding_side: {tokenizer.padding_side}")

In [ ]:
# Cell 8: Load and validate dataset
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if line:
                try: records.append(json.loads(line))
                except json.JSONDecodeError as e: print(f"  Line {i}: {e}")
    return records

def validate_record(rec, idx):
    out = rec.get("output", {})
    if isinstance(out, str):
        try: out = json.loads(out)
        except: return False
    return (out.get("intent") in VALID_INTENTS
            and out.get("scope") in VALID_SCOPES
            and bool(rec.get("input")))

print(f"Loading {TRAIN_FILE}...")
all_records     = load_jsonl(TRAIN_FILE)
valid_records   = [r for i, r in enumerate(all_records) if validate_record(r, i)]
multitask_count = sum(1 for r in valid_records if r.get("output", {}).get("rewritten_query"))
print(f"  Total: {len(valid_records)} / {len(all_records)} valid")
print(f"  Routing-only: {len(valid_records) - multitask_count}  |  With rewrite: {multitask_count}")

print(f"Loading {VAL_FILE}...")
val_records = [r for i, r in enumerate(load_jsonl(VAL_FILE)) if validate_record(r, i)]
print(f"  Val: {len(val_records)}")

intent_counts = Counter()
for r in valid_records:
    out = r.get("output", {})
    if isinstance(out, str): out = json.loads(out)
    intent_counts[out.get("intent", "?")] += 1
print("\nIntent distribution:")
for k, v in sorted(intent_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} {v:>4} ({v/len(valid_records)*100:4.1f}%)")


In [ ]:
# Cell 10: Format data — manual ChatML (no thinking tokens in training data)
# Avoids apply_chat_template which injects <think></think> even with enable_thinking=False

IM_START = '<|im_start|>'
IM_END   = '<|im_end|>'
NL       = chr(10)  # newline — use chr(10) to avoid backslash-n in notebook JSON

def format_record(rec):
    user_msg = str(rec.get('input', '')).strip()
    ctx = rec.get('context')
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(',', ':'))
        user_msg = '[CONTEXT: ' + ctx_str + ']' + NL + user_msg

    out = rec.get('output', {})
    if isinstance(out, str): out = json.loads(out)
    output_dict = {
        'intent':     out['intent'],
        'scope':      out['scope'],
        'confidence': round(float(out.get('confidence', 0.9)), 2),
    }
    rw = out.get('rewritten_query')
    if rw and str(rw).strip():
        output_dict['rewritten_query'] = str(rw).strip()

    # Manual ChatML — clean format, no <think> tokens
    text  = IM_START + 'system'    + NL + SYSTEM_PROMPT + IM_END + NL
    text += IM_START + 'user'      + NL + user_msg      + IM_END + NL
    text += IM_START + 'assistant' + NL + json.dumps(output_dict, ensure_ascii=False) + IM_END + NL
    return text

print('Formatting data...')
train_texts = [format_record(r) for r in valid_records]
val_texts   = [format_record(r) for r in val_records]

print('Sample:')
print('-' * 60)
print(train_texts[0])
print('-' * 60)
lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print('Lengths (first 200): min=' + str(min(lengths)) + ', max=' + str(max(lengths)) + ', avg=' + str(sum(lengths)//len(lengths)))
print('Over limit: ' + str(sum(1 for l in lengths if l > MAX_SEQ_LENGTH)) + '/200')

In [ ]:
# Cell 10: HuggingFace datasets — pre-tokenize with explicit truncation
# Avoids SFTConfig input/label length mismatch bug in TRL 0.17+
raw_train = Dataset.from_dict({"text": train_texts})
raw_val   = Dataset.from_dict({"text": val_texts})

def tokenize_fn(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_tensors=None,
    )
    enc["labels"] = [ids[:] for ids in enc["input_ids"]]  # causal LM: labels = input_ids
    return enc

train_dataset = raw_train.map(tokenize_fn, batched=True, remove_columns=["text"])
val_dataset   = raw_val.map(tokenize_fn,   batched=True, remove_columns=["text"])

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Columns: {train_dataset.column_names}")
# Expected: [input_ids, attention_mask, labels]


In [ ]:
# Cell 11: Data collator — pads input_ids AND labels to same length per batch
# label_pad_token_id=-100: padding positions ignored in cross_entropy loss
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,   # efficient on A100 tensor cores
)
print("DataCollatorForSeq2Seq ready")


In [ ]:
# Cell 12: Training arguments
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Schedule ──
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,   # 8 on A100
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,   # effective batch = 32
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",

    # ── Precision ──
    fp16=False,
    bf16=True,   # A100 native bf16

    # ── Logging / Checkpointing ──
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── Optimizer ──
    optim="adamw_torch_fused",   # best for bf16 on A100
    weight_decay=0.01,
    max_grad_norm=1.0,

    # ── Misc ──
    seed=42,
    report_to="none",
    dataloader_num_workers=4,
)
prec = "bf16" if training_args.bf16 else "fp16"
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Precision:    {prec}")
print(f"Eff. batch:   {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps/epoch:  {steps_per_epoch}  |  Total steps: {steps_per_epoch * EPOCHS}")


In [ ]:
# Cell 13: SFT Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,   # explicit collator — input+labels always same shape
    args=training_args,
    packing=False,
)
print("Trainer ready")
vram  = torch.cuda.memory_reserved(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM reserved: {vram:.1f} GB / {total:.1f} GB")


In [ ]:
# Cell 14: TRAIN (~45–60 min on A100 40GB)
print("=" * 60)
print(f"Model:   {BASE_MODEL}")
print(f"LoRA:    r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Train:   {len(train_dataset)} ({multitask_count} with-rewrite)")
print(f"Epochs:  {EPOCHS}")
print("=" * 60)

train_result = trainer.train()

print(f"\nTrain loss: {train_result.training_loss:.4f}")
print(f"Runtime:    {train_result.metrics['train_runtime']:.0f}s  ({train_result.metrics['train_runtime']/60:.1f} min)")
print(f"Samples/s:  {train_result.metrics['train_samples_per_second']:.1f}")


In [ ]:
# Cell 15: Save LoRA adapter to Google Drive
adapter_dir = Path(OUTPUT_DIR) / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved: {adapter_dir}")
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")


In [ ]:
# Cell 16: Full evaluation on multitask_val.jsonl
print("=" * 60)
print("Running full evaluation (~10 min on A100)...")
print("=" * 60)

# Use model.eval() — NOT FastLanguageModel.for_inference()
# Reason: Unsloth Qwen3 fast-inference has KV-cache RoPE shape mismatch bug
model.eval()

val_examples = [
    json.loads(l)
    for l in Path(VAL_FILE).read_text(encoding="utf-8").splitlines()
    if l.strip()
]
print(f"Loaded {len(val_examples)} val examples")

r_routing, r_rewrite, r_norewrite = [], [], []
entity_hits = 0; rw_present = 0; norewrite_ok = 0

for i, ex in enumerate(val_examples):
    ctx      = ex.get("context")
    gt       = ex["output"]
    gt_rw    = gt.get("rewritten_query")
    user_msg = str(ex["input"])
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",", ":"))
        user_msg = f"[CONTEXT: {ctx_str}]\n{user_msg}"

    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg}]
    try:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(txt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=150, do_sample=False,
            use_cache=False)   # avoids Unsloth Qwen3 KV-cache shape bug
    raw = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    # Strip Qwen3 thinking block: <think>...</think> may appear even with enable_thinking=False
    if "</think>" in raw:
        raw = raw[raw.rfind("</think>") + len("</think>"):].strip()
    resp = raw

    try:    pred = json.loads(resp)
    except: pred = {}

    intent_ok = (pred.get("intent") == gt.get("intent"))
    scope_ok  = (pred.get("scope")  == gt.get("scope"))
    ok_both   = intent_ok and scope_ok
    pred_rw   = pred.get("rewritten_query") or None

    if ctx is None:
        r_routing.append(ok_both)
    elif gt_rw:
        r_rewrite.append(ok_both)
        if pred_rw:
            rw_present += 1
            loc = ctx.get("location", "")
            if loc and loc.lower() in pred_rw.lower():
                entity_hits += 1
    else:
        r_norewrite.append(ok_both)
        if pred_rw is None:
            norewrite_ok += 1

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(val_examples)}")

def pct(a, b): return f"{a}/{b} = {a/b*100:.1f}%" if b else "N/A"
n_rw = len(r_rewrite)   or 1
n_nr = len(r_norewrite) or 1

print("\n" + "=" * 60)
print("FULL EVAL RESULTS")
print("=" * 60)
print(f"Routing accuracy    : {pct(sum(r_routing),  len(r_routing))}    target >= 90%")
print(f"Rewrite routing acc : {pct(sum(r_rewrite),  n_rw)}    target >= 85%")
print(f"No-rewrite routing  : {pct(sum(r_norewrite),n_nr)}    target >= 80%")
print(f"--- rewrite quality ---")
print(f"Rewrite presence    : {pct(rw_present,  n_rw)}")
print(f"Entity coverage     : {pct(entity_hits, n_rw)}    target >= 70%")
print(f"No-rewrite (absent) : {pct(norewrite_ok,n_nr)}")
print("=" * 60)

PASS_ROUTING = (sum(r_routing) / len(r_routing)) >= 0.90 if r_routing else False
PASS_ENTITY  = (entity_hits / n_rw) >= 0.70
print(f"\nPass routing>=90% : {'PASS' if PASS_ROUTING else 'FAIL'}")
print(f"Pass entity>=70%  : {'PASS' if PASS_ENTITY else 'FAIL'}")
if PASS_ROUTING and PASS_ENTITY:
    print("-> Ready to deploy to Ollama")
else:
    print("-> Review before deploy")


In [ ]:
# Cell 17: Inference test — 6 manual cases
model.eval()

TEST_CASES = [
    ("Routing basic",
     "Bay gio thoi tiet Cau Giay the nao?",
     "current_weather", None),
    ("Forecast",
     "Cuoi tuan Ha Noi the nao?",
     "daily_forecast", None),
    ("Rewrite — location carry",
     '[CONTEXT: {"location":"Cau Giay","intent":"current_weather","turn":1}]\nCon ngay mai?',
     "daily_forecast", "Cau Giay"),
    ("Rewrite — pronoun",
     '[CONTEXT: {"location":"Hoang Mai","intent":"rain_query","turn":1}]\nO do co gio khong?',
     "wind_query", "Hoang Mai"),
    ("No-rewrite — explicit location",
     '[CONTEXT: {"location":"Cau Giay","intent":"current_weather","turn":1}]\nThoi tiet Dong Da ngay mai?',
     "daily_forecast", None),
    ("Activity",
     "Sang mai di chay bo duoc khong?",
     "activity_weather", None),
]

print("=" * 70)
print("INFERENCE TEST")
print("=" * 70)
all_ok = True

for desc, user_msg, exp_intent, exp_entity in TEST_CASES:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg}]
    try:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(txt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=150, do_sample=False, use_cache=False)
    raw = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw[raw.rfind("</think>") + len("</think>"):].strip()
    resp = raw

    ok_intent = False; ok_entity = True; pred_intent = "?"; pred_rw = None
    try:
        parsed      = json.loads(resp)
        pred_intent = parsed.get("intent", "?")
        pred_rw     = parsed.get("rewritten_query")
        ok_intent   = (pred_intent == exp_intent)
        if exp_entity:
            ok_entity = bool(pred_rw) and exp_entity.lower() in pred_rw.lower()
        else:
            ok_entity = (pred_rw is None)
    except Exception:
        ok_intent = False

    ok = ok_intent and ok_entity
    if not ok: all_ok = False

    print(f"\n{'OK' if ok else 'FAIL'} [{desc}]")
    print(f"   Input:   {user_msg[:80]}")
    print(f"   Intent:  {pred_intent!r}  (expected {exp_intent!r})  {'ok' if ok_intent else 'FAIL'}")
    if exp_entity:
        print(f"   Entity:  {exp_entity!r} in rewrite: {'ok' if ok_entity else 'FAIL'}")
        print(f"   Rewrite: {pred_rw}")
    else:
        print(f"   No-rewrite: {'ok' if ok_entity else 'FAIL'} (rewritten_query absent)")

print(f"\n{'='*70}")
print(f"All passed: {all_ok}")


In [ ]:
# Cell 18: Export GGUF (Q8_0 for production + Q4_K_M as backup)
import subprocess, shutil, sys

LLAMA_CPP_DIR = "/tmp/llama.cpp"

if not Path(LLAMA_CPP_DIR).exists():
    print("Cloning llama.cpp...")
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/ggml-org/llama.cpp.git",
                    LLAMA_CPP_DIR], check=True)

print("Building llama.cpp with CMake...")
build_dir = f"{LLAMA_CPP_DIR}/build"
subprocess.run(["cmake", LLAMA_CPP_DIR, "-B", build_dir,
                "-DBUILD_SHARED_LIBS=OFF", "-DGGML_CUDA=OFF"], check=True)
subprocess.run(["cmake", "--build", build_dir, "--config", "Release",
                "-j", str(os.cpu_count() or 4),
                "--target", "llama-quantize"], check=True)

# Merge LoRA into full model
MERGED_DIR = "/tmp/merged_model"
print("Merging LoRA into bf16 model...")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# Convert HF -> GGUF f16
print("Converting HF -> GGUF f16...")
converter = f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py"
f16_gguf  = "/tmp/model-f16.gguf"
subprocess.run([sys.executable, converter, MERGED_DIR,
                "--outfile", f16_gguf, "--outtype", "f16"], check=True)

# Quantize to Q8_0 (production) and Q4_K_M (backup)
quantizer = f"{build_dir}/bin/llama-quantize"
for qtype in ["q8_0", "q4_k_m"]:
    out_dir  = Path(GGUF_DIR) / qtype
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = str(out_dir / f"unsloth.{qtype.upper()}.gguf")
    print(f"Quantizing {qtype}...")
    subprocess.run([quantizer, f16_gguf, out_path, qtype], check=True)

os.remove(f16_gguf)
shutil.rmtree(MERGED_DIR, ignore_errors=True)

print("\nGGUF files saved to Google Drive:")
for f in sorted(Path(GGUF_DIR).rglob("*.gguf")):
    print(f"  {f.relative_to(GGUF_DIR)}  ({f.stat().st_size/1e9:.2f} GB)")


In [ ]:
# Cell 19: Generate Ollama Modelfile
gguf_name = "unsloth.Q8_0.gguf"

lines = [
    "FROM ./" + gguf_name,
    "",
    'SYSTEM "' + SYSTEM_PROMPT.replace('"', '\\"') + '"',
    "",
    "PARAMETER temperature 0",
    "PARAMETER num_predict 128",
    'PARAMETER stop "<|im_end|>"',
    'PARAMETER stop "<|endoftext|>"',
]
modelfile_content = "\n".join(lines)

mf = Path(GGUF_DIR) / "q8_0" / "Modelfile"
mf.write_text(modelfile_content, encoding="utf-8")
print(f"Modelfile saved: {mf}")
print("\nDeploy on local machine:")
print("  1. Download q8_0/ folder from Google Drive")
print(f"  2. cd into folder containing {gguf_name} and Modelfile")
print("  3. ollama create hanoi-weather-router -f Modelfile")
print("  4. Test: ollama run hanoi-weather-router")
print("\n.env: OLLAMA_MODEL=hanoi-weather-router  USE_SLM_ROUTER=true")


In [ ]:
# Cell 20: (Optional) Push LoRA adapter to HuggingFace Hub
# Add HF_TOKEN in Colab Secrets: Runtime > Manage secrets (key icon in sidebar)
PUSH_TO_HUB  = True
HUB_MODEL_ID = "daredevil467/qwen3-1.7b-hanoi-weather-router-v2"

if PUSH_TO_HUB:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = os.getenv("HF_TOKEN", "")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        model.push_to_hub(HUB_MODEL_ID, token=hf_token, private=True)
        tokenizer.push_to_hub(HUB_MODEL_ID, token=hf_token, private=True)
        print(f"Pushed: https://huggingface.co/{HUB_MODEL_ID}")
    else:
        print("HF_TOKEN not found.")
        print("Add it: Runtime > Manage secrets > Add new secret: HF_TOKEN")
else:
    print("Hub push skipped (set PUSH_TO_HUB=True to enable)")


In [ ]:
# Cell 21: Summary
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Base:      {BASE_MODEL}")
print(f"LoRA:      r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Train:     {len(train_dataset)} ({multitask_count} with-rewrite)")
print(f"Loss:      {train_result.training_loss:.4f}")
print(f"\nOutput files in Google Drive ({DRIVE_DIR}):")
for root, dirs, files in os.walk(str(DRIVE_DIR)):
    for fname in sorted(files):
        fp = os.path.join(root, fname)
        sz = os.path.getsize(fp) / 1e6
        if sz > 0.1:
            rel = os.path.relpath(fp, str(DRIVE_DIR))
            print(f"  {rel:<55} {sz:7.1f} MB")
